In [ ]:
import os
import sys
## adding the root directory to the library directory so still can read utility functions
sys.path.append(os.path.abspath(os.path.join(os.path.dirname("..."), "..")))
project_root = os.path.abspath(os.path.expanduser("..."))
sys.path.append(project_root)
os.chdir(project_root) # setting work directory

import matplotlib.pyplot as plt
import warnings
import pickle 
import pandas as pd
import numpy as np

from sklearn.datasets import make_regression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
import lightgbm as lgb


def fxn():
    warnings.warn("deprecated", DeprecationWarning)

with warnings.catch_warnings():
    
    warnings.simplefilter("ignore")
    fxn()

In [ ]:
### reading in CHS data
chs_raw = pd.read_csv("Data/final_datasets/CHS_SOMA_log2scaled_nodup_final.tsv.gz", sep = "\t", 
                      compression = "gzip").drop(columns={"bmi"}, axis = 1)

chs_raw.head(5)

In [ ]:
### read in MESA soma for the column names
MESA_soma = pd.read_csv("Data/final_datasets/MESA_SOMA_5k_log2scaled_nodup_final.tsv.gz", 
                        sep = "\t", compression = "gzip").drop(columns={"bmi"})
MESA_soma["LEP"].describe()
MESA_olink = pd.read_csv("Data/final_datasets/MESA_Olink_3k_log2_nodup_rm10na_knn_final.tsv.gz", 
                         sep = "\t", compression = "gzip")


# Find subjects who are in both datasets

overlap_IDs = np.unique(list(set(np.array(MESA_olink["id"])).intersection(set(np.array(MESA_soma["id"])))))


MESA_olink  = MESA_olink[MESA_olink["id"].isin(overlap_IDs)].sort_values(by=["id"]).drop_duplicates(["id"])
MESA_soma   = MESA_soma[MESA_soma["id"].isin(overlap_IDs)].sort_values(by=["id"]).drop_duplicates(["id"])

In [ ]:
# common columns of the two dataset
common_cols = [col for col in chs_raw.columns if col in MESA_soma.columns]

print(len(common_cols))


chs_short = chs_raw[common_cols]
chs_short.head(5)

In [ ]:
MESA_soma = MESA_soma.set_index("id")
MESA_olink = MESA_olink.set_index("id")

In [ ]:
MESA_soma['LEP'].describe()

In [ ]:
chs_short['LEP'].describe()
chs_short = chs_short.set_index("id")

In [ ]:
chs_soma = chs_short

In [ ]:
protlist = MESA_olink.columns

print(protlist[1:5])
print(len(protlist))

## Evaluating Protein performance

In [ ]:
%%time
results = []


from scipy.stats import pearsonr
soma_cols = set(MESA_soma.columns)   # what exists in SOMAscan

train_index = pickle.load(open("Data/helper_data/train_index.pkl", "rb"))
test_index = pickle.load(open("Data/helper_data/test_index.pkl", "rb"))


for protein_name in protlist:
    model_path = f"Data/soma_trained_models/soma_olink/5k_log2scaled_soma_to_3kolink/{protein_name}_hgb.pkl"
    reg = pickle.load(open(model_path, "rb"))
    print(f"load the model for {protein_name}")
    

    if protein_name in soma_cols:
        
        ## loading test set, predicted value and actual value
        X_protein = MESA_soma[protein_name]
        X_train_protein = X_protein.loc[train_index]
        X_test_protein = X_protein.loc[test_index]
        y = MESA_olink[protein_name]
        
        X_train = MESA_soma.loc[train_index]
        y_train = y.loc[train_index]
        
        X_test = MESA_soma.loc[test_index]
        y_test = y.loc[test_index]
        
        y_pred = reg.predict(X_test)
        
        # correlation between SOMA and OLINK full column
        pretrain_r, _ = pearsonr(y_train, X_train_protein)
        pretrain_r2 = pretrain_r **2
        
        # pred and observed comparison
        model_r, _ = pearsonr(y_test, y_pred)
        model_r2 = model_r**2
        
        ## post training correlation?
        posttrain_r,_ = pearsonr(y_pred, X_test_protein)
        posttrain_r2 = posttrain_r ** 2
        
        testset_r, _ = pearsonr(y_test, X_test_protein)
        testset_r2 = testset_r ** 2
        
        results.append({
            "protein": protein_name,
            "has_soma": protein_name in soma_cols,
            "model_r": model_r,
            "model_r2": model_r2,
            "pretrain_r": pretrain_r,
            "pretrain_rsq": pretrain_r2,
            "posttrain_r": posttrain_r,
            "posttrain_rsq": posttrain_r2,
            "testset_r": testset_r,
            "testset_r2": testset_r2
        })
    
    else:
        ## we can still calculate R2 for pred and actual
        ## loading test set, predicted value and actual value
        y = MESA_olink[protein_name]
        
        X_test = MESA_soma.loc[test_index]
        y_test = y.loc[test_index]
        
        y_pred = reg.predict(X_test)
        
        ## r2
        model_r, _ = pearsonr(y_test, y_pred) # no output pvalue
        model_r2 = model_r**2
        
        results.append({
            "protein": protein_name,
            "has_soma": protein_name in soma_cols,
            "model_r": model_r,
            "model_r2": model_r2,
            "pretrain_r": np.nan,
            "pretrain_rsq": np.nan,
            "posttrain_r": np.nan,
            "posttrain_rsq": np.nan,
            "testset_r": np.nan,
            "testset_r2": np.nan
        })
    

In [ ]:
results

In [ ]:
results_df = pd.DataFrame(results)
rsq_highest_row = results_df.loc[results_df["model_r2"].idxmax()]

## mean of rsq for all the proteins
print(f'Mean R square among proteins is: {results_df["model_r2"].mean():.4f}')
print(f'Max R square among proteins is: {results_df["model_r2"].max():.4f} for protein: {rsq_highest_row["protein"]}')

# correlation is between the soma measurement and olink measurement
print(f'Mean correlation among proteins is: {results_df["pretrain_rsq"].mean():.4f}')
print(f'Max correlation: {results_df["pretrain_rsq"].max():.4f} for protein: {rsq_highest_row["protein"]}')

In [ ]:
results_df.loc[results_df["protein"] == "LEP"]

In [ ]:
### saving results

results_df.to_csv("Results/metrics/HGB_log2scaled_5ksoma_3kolink_results.tsv", sep = "\t", index = False)

## Predicting CHS Olink

In [ ]:
%%time

# ## validation section
# from scipy.stats import pearsonr

results = []

for protein_name in protlist:

    model_path = f"Data/soma_trained_models/soma_olink/5k_log2scaled_soma_3kolink/{protein_name}_hgb.pkl"
    reg = pickle.load(open(model_path, "rb"))

    print(f"load the model for {protein_name}")

    chs_soma_short = chs_soma

    y_pred = reg.predict(chs_soma_short)

    # temporary result table
    result = pd.DataFrame({
        "id": chs_soma.index,
        protein_name: y_pred
    })

    # appending result
    results.append(result)

In [ ]:
## indexing
indexed = [df.set_index("id") for df in results]

chs_results = pd.concat(indexed, axis = 1, join = "inner").reset_index()

In [ ]:
chs_results

In [ ]:
chs_results.to_csv("Results/CHS_pred/HGB_predicted_CHS_log2scaled_5ksoma_3kolink.tsv.gz", sep = "\t", index = False, compression="gzip")